In [1]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import parselmouth
import noisereduce as nr
import soundfile as sf
from parselmouth.praat import call

# --- CONFIGURATION ---
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/" 
OUTPUT_CSV = "../data/processed/bio_markers_refined.csv"

LABEL_MAP = {
    'hungry': 0, 'belly pain': 1, 'cold_hot': 1, 'discomfort': 1,
    'tired': 2, 'burping': 2, 'lonely': 2, 'scared': 2
}

def extract_ultimate_features(audio_path):
    try:
        # 1. Load Audio
        y, sr = librosa.load(audio_path, sr=22050)
        
        # 2. Moderate Noise Reduction (Essential for accurate MFCCs)
        try:
            y_clean = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.6)
        except:
            y_clean = y 

        # Save temp for Parselmouth
        temp_path = "temp_clean.wav"
        sf.write(temp_path, y_clean, sr)

        # --- CLINICAL FEATURES (For Report) ---
        sound = parselmouth.Sound(temp_path)
        pitch = sound.to_pitch()
        f0 = pitch.selected_array['frequency']
        f0 = f0[f0 > 100]
        
        if len(f0) > 5:
            pitch_max = np.max(f0)
            pitch_mean = np.mean(f0)
        else:
            pitch_max, pitch_mean = 0, 0

        # Jitter/Shimmer
        try:
            point_process = call(sound, "To PointProcess (periodic, cc)", 75, 600)
            jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        except:
            jitter, shimmer = 0, 0

        features = {
            "clin_pitch_max": pitch_max,
            "clin_pitch_mean": pitch_mean,
            "clin_jitter": jitter,
            "clin_shimmer": shimmer,
        }

        # --- DEEP AUDIO FEATURES (To Fix Low Accuracy) ---
        # MFCCs (40) - The "Fingerprint" of the sound
        mfccs = librosa.feature.mfcc(y=y_clean, sr=sr, n_mfcc=40)
        features.update({f"mfcc_mean_{i}": np.mean(e) for i, e in enumerate(mfccs)})
        features.update({f"mfcc_std_{i}": np.std(e) for i, e in enumerate(mfccs)})

        # Mel Spectrogram (128) - The "Image" of the sound
        mel = librosa.feature.melspectrogram(y=y_clean, sr=sr, n_mels=128)
        features.update({f"mel_mean_{i}": np.mean(e) for i, e in enumerate(mel)})
        
        # Contrast & Chroma
        contrast = librosa.feature.spectral_contrast(S=np.abs(librosa.stft(y_clean)), sr=sr)
        features.update({f"contrast_mean_{i}": np.mean(e) for i, e in enumerate(contrast)})
        
        chroma = librosa.feature.chroma_stft(y=y_clean, sr=sr)
        features.update({f"chroma_mean_{i}": np.mean(e) for i, e in enumerate(chroma)})

        return features

    except Exception as e:
        return None

# --- MAIN LOOP ---
data_list = []
print("🚀 Starting Ultimate Extraction...")

for folder, label in LABEL_MAP.items():
    path = os.path.join(DATASET_PATH, folder)
    files = glob.glob(os.path.join(path, "*.wav"))
    print(f"   Processing {folder}...")
    
    for file in files:
        feats = extract_ultimate_features(file)
        if feats:
            feats['label'] = label
            data_list.append(feats)

df = pd.DataFrame(data_list)
df.fillna(0, inplace=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ DONE! Saved {len(df)} samples to {OUTPUT_CSV}")

f:\Research\Project\Final\infant-growth-monitoring-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starting Ultimate Extraction...
   Processing hungry...
   Processing belly pain...
   Processing cold_hot...
   Processing discomfort...
   Processing tired...
   Processing burping...
   Processing lonely...
   Processing scared...
✅ DONE! Saved 1095 samples to ../data/processed/bio_markers_refined.csv
